In [4]:
!pip install sentence-transformers scikit-learn

In [3]:
!git clone https://github.com/encode/httpx.git

Cloning into 'httpx'...
remote: Enumerating objects: 10983, done.
remote: Total 10983 (delta 0), reused 0 (delta 0), pack-reused 10983 (from 1)
Receiving objects: 100% (10983/10983), 8.41 MiB | 20.71 MiB/s, done.
Resolving deltas: 100% (8278/8278), done.


In [5]:
%cd httpx

/content/httpx


In [6]:
!git checkout b5addb64f0161ff6bfe94c124ef76f6a1fba5254

Note: switching to 'b5addb64f0161ff6bfe94c124ef76f6a1fba5254'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at b5addb6 Adapt test_response_decode_text_using_autodetect for chardet 6.0 (#3773)


In [7]:
from pathlib import Path

docs_path = Path("docs")

arquivos_md = list(docs_path.rglob("*.md"))

print("Quantidade de arquivos:", len(arquivos_md))

for arquivo in arquivos_md:
    print(arquivo)

Quantidade de arquivos: 23
docs/compatibility.md
docs/index.md
docs/api.md
docs/quickstart.md
docs/environment_variables.md
docs/troubleshooting.md
docs/logging.md
docs/code_of_conduct.md
docs/contributing.md
docs/async.md
docs/exceptions.md
docs/third_party_packages.md
docs/http2.md
docs/advanced/ssl.md
docs/advanced/text-encodings.md
docs/advanced/event-hooks.md
docs/advanced/proxies.md
docs/advanced/extensions.md
docs/advanced/timeouts.md
docs/advanced/resource-limits.md
docs/advanced/clients.md
docs/advanced/transports.md
docs/advanced/authentication.md


In [8]:
documentos = []

for arquivo in arquivos_md:
    texto = arquivo.read_text(encoding="utf-8")

    documentos.append({
        "arquivo": str(arquivo),
        "texto": texto
    })

print("Documentos carregados:", len(documentos))

Documentos carregados: 23


In [17]:
def criar_chunks(texto, tamanho=80, overlap=20):
    palavras = texto.split()

    chunks = []

    inicio = 0

    while inicio < len(palavras):
        fim = inicio + tamanho

        chunk = " ".join(palavras[inicio:fim])

        chunks.append(chunk)

        inicio += tamanho - overlap

    return chunks

In [18]:
def criar_chunks(texto, tamanho=80, overlap=20):
    palavras = texto.split()

    chunks = []

    inicio = 0

    while inicio < len(palavras):
        fim = inicio + tamanho

        chunk = " ".join(palavras[inicio:fim])

        chunks.append(chunk)

        inicio += tamanho - overlap

    return chunks

In [20]:
def criar_chunks(texto, tamanho=80, overlap=20):
    palavras = texto.split()

    chunks = []

    inicio = 0

    while inicio < len(palavras):
        fim = inicio + tamanho

        chunk = " ".join(palavras[inicio:fim])

        chunks.append(chunk)

        inicio += tamanho - overlap

    return chunks

In [27]:
texto_teste = documentos[0]["texto"]

chunks_teste = criar_chunks(texto_teste)

print("Quantidade de chunks:", len(chunks_teste))

print("\nPrimeiro chunk:")
print(chunks_teste[0])


Quantidade de chunks: 22

Primeiro chunk:
# Requests Compatibility Guide HTTPX aims to be broadly compatible with the `requests` API, although there are a few design differences in places. This documentation outlines places where the API differs... ## Redirects Unlike `requests`, HTTPX does **not follow redirects by default**. We differ in behaviour here [because auto-redirects can easily mask unnecessary network calls being made](https://github.com/encode/httpx/discussions/1785). You can still enable behaviour to automatically follow redirects, but you need to do so explicitly... ```python response = client.get(url, follow_redirects=True) ``` Or


In [28]:
chunks = []

for documento in documentos:

    partes = criar_chunks(documento["texto"])

    for i, parte in enumerate(partes):

        chunks.append({
            "texto": parte,
            "arquivo": documento["arquivo"],
            "chunk_id": i
        })

print("Total de chunks:", len(chunks))

print(chunks[0])

Total de chunks: 277
{'texto': '# Requests Compatibility Guide HTTPX aims to be broadly compatible with the `requests` API, although there are a few design differences in places. This documentation outlines places where the API differs... ## Redirects Unlike `requests`, HTTPX does **not follow redirects by default**. We differ in behaviour here [because auto-redirects can easily mask unnecessary network calls being made](https://github.com/encode/httpx/discussions/1785). You can still enable behaviour to automatically follow redirects, but you need to do so explicitly... ```python response = client.get(url, follow_redirects=True) ``` Or', 'arquivo': 'docs/compatibility.md', 'chunk_id': 0}


In [33]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [39]:
textos = [chunk["texto"] for chunk in chunks]

embeddings = modelo.encode(
    textos,
    normalize_embeddings=True
)

print(embeddings.shape)

(277, 384)


In [40]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(pergunta, top_k=3):

    if not pergunta.strip():
        print("Digite uma pergunta.")
        return

    if top_k <= 0:
        print("top_k deve ser maior que zero.")
        return

    embedding_pergunta = modelo.encode(
        [pergunta],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        embedding_pergunta,
        embeddings
    )[0]

    indices = scores.argsort()[::-1][:top_k]

    for posicao, indice in enumerate(indices, start=1):

        resultado = chunks[indice]

        print("=" * 70)
        print(f"Resultado {posicao}")
        print(f"Score: {scores[indice]:.4f}")
        print(f"Arquivo: {resultado['arquivo']}")
        print(f"Chunk: {resultado['chunk_id']}")
        print()
        print(resultado["texto"])

In [41]:
buscar("Como fazer uma requisição GET usando HTTPX?")

Resultado 1
Score: 0.6443
Arquivo: docs/advanced/proxies.md
Chunk: 0

HTTPX supports setting up [HTTP proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`. <div align="center"> <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Open_proxy_h2g2bob.svg/480px-Open_proxy_h2g2bob.svg.png"/> <figcaption><em>Diagram of how a proxy works (source: Wikipedia). The left hand side "Internet" blob may be your HTTPX client requesting <code>example.com</code> through a proxy.</em></figcaption> </div> ## HTTP Proxies To route all traffic (HTTP and HTTPS) to a proxy located at `http://localhost:8030`, pass the proxy URL to the client... ```python with httpx.Client(proxy="http://localhost:8030") as client:
Resultado 2
Score: 0.6347
Arquivo: docs/compatibility.md
Chunk: 18

HTTP networking code to the excellent [`urllib3` library](https://urllib3.r

In [42]:
buscar("How do I make a GET request with HTTPX?")

Resultado 1
Score: 0.5929
Arquivo: docs/api.md
Chunk: 5

`Request` *An HTTP request. Can be constructed explicitly for more control over exactly what gets sent over the wire.* ```pycon >>> request = httpx.Request("GET", "https://example.org", headers={'host': 'example.org'}) >>> response = client.send(request) ``` * `def __init__(method, url, [params], [headers], [cookies], [content], [data], [files], [json], [stream])` * `.method` - **str** * `.url` - **URL** * `.content` - **byte**, **byte iterator**, or **byte async iterator** * `.headers` - **Headers** * `.cookies` - **Cookies** ## `URL` *A normalized, IDNA supporting URL.* ```pycon >>> url
Resultado 2
Score: 0.5497
Arquivo: docs/quickstart.md
Chunk: 0

# QuickStart First, start by importing HTTPX: ```pycon >>> import httpx ``` Now, let’s try to get a webpage. ```pycon >>> r = httpx.get('https://httpbin.org/get') >>> r <Response [200 OK]> ``` Similarly, to make an HTTP POST request: ```pycon >>> r = httpx.post('https://httpbin.org/

In [43]:
buscar("What can HTTPX be used for?")

Resultado 1
Score: 0.6932
Arquivo: docs/index.md
Chunk: 5

[HTTP/2](http2.md) section. The [Developer Interface](api.md) provides a comprehensive API reference. To find out about tools that integrate with HTTPX, see [Third Party Packages](third_party_packages.md). ## Dependencies The HTTPX project relies on these excellent libraries: * `httpcore` - The underlying transport implementation for `httpx`. * `h11` - HTTP/1.1 support. * `certifi` - SSL certificates. * `idna` - Internationalized domain name support. * `sniffio` - Async library autodetection. As well as these optional installs: * `h2` - HTTP/2 support. *(Optional, with `httpx[http2]`)* *
Resultado 2
Score: 0.6834
Arquivo: docs/third_party_packages.md
Chunk: 0

# Third Party Packages As HTTPX usage grows, there is an expanding community of developers building tools and libraries that integrate with HTTPX, or depend on HTTPX. Here are some of them. <!-- NOTE: Entries are alphabetised. --> ## Plugins ### Hishel [GitHub](https://gi

In [44]:
buscar("Quem foi o campeão brasileiro de futebol em 2024?")

Resultado 1
Score: 0.2726
Arquivo: docs/advanced/extensions.md
Chunk: 4

[(b'Age', b'553715'), (b'Cache-Control', b'max-age=604800'), (b'Content-Type', b'text/html; charset=UTF-8'), (b'Date', b'Thu, 21 Oct 2021 17:08:42 GMT'), (b'Etag', b'"3147526947+ident"'), (b'Expires', b'Thu, 28 Oct 2021 17:08:42 GMT'), (b'Last-Modified', b'Thu, 17 Oct 2019 07:18:26 GMT'), (b'Server', b'ECS (nyb/1DCD)'), (b'Vary', b'Accept-Encoding'), (b'X-Cache', b'HIT'), (b'Content-Length', b'1256')])} # http11.receive_response_body.started {'request': <Request [b'GET']>} # http11.receive_response_body.complete {'return_value': None} # http11.response_closed.started {} # http11.response_closed.complete {'return_value': None} ``` The `event_name` and `info` arguments here will be one of the following: * `{event_type}.{event_name}.started`, `<dictionary of keyword arguments>` * `{event_type}.{event_name}.complete`, `{"return_value": <...>}` * `{event_type}.{event_name}.failed`,
Resultado 2
Score: 0.1596
Arquivo: do

In [45]:
buscar("")

Digite uma pergunta.


In [49]:
buscar("What is HTTPX?", top_k=0)

top_k deve ser maior que zero.


In [50]:
import re

def extrair_titulo(texto):
    for linha in texto.splitlines():
        linha = linha.strip()

        if linha.startswith("# "):
            return linha[2:].strip()

    return "Sem título"

In [51]:
chunks = []

for documento in documentos:

    partes = criar_chunks(documento["texto"])
    titulo = extrair_titulo(documento["texto"])

    for i, parte in enumerate(partes):

        chunks.append({
            "texto": parte,
            "arquivo": documento["arquivo"],
            "chunk_id": i,
            "titulo": titulo
        })

print("Total de chunks:", len(chunks))
print(chunks[0])

Total de chunks: 277
{'texto': '# Requests Compatibility Guide HTTPX aims to be broadly compatible with the `requests` API, although there are a few design differences in places. This documentation outlines places where the API differs... ## Redirects Unlike `requests`, HTTPX does **not follow redirects by default**. We differ in behaviour here [because auto-redirects can easily mask unnecessary network calls being made](https://github.com/encode/httpx/discussions/1785). You can still enable behaviour to automatically follow redirects, but you need to do so explicitly... ```python response = client.get(url, follow_redirects=True) ``` Or', 'arquivo': 'docs/compatibility.md', 'chunk_id': 0, 'titulo': 'Requests Compatibility Guide'}


In [53]:
textos = [chunk["texto"] for chunk in chunks]

embeddings = modelo.encode(
    textos,
    normalize_embeddings=True
)

print(embeddings.shape)

(277, 384)


In [54]:
chunks = []

for documento in documentos:

    partes = criar_chunks(documento["texto"])
    titulo = extrair_titulo(documento["texto"])

    for i, parte in enumerate(partes):

        chunks.append({
            "texto": parte,
            "arquivo": documento["arquivo"],
            "chunk_id": i,
            "titulo": titulo
        })

print("Total de chunks:", len(chunks))
print(chunks[0])

Total de chunks: 277
{'texto': '# Requests Compatibility Guide HTTPX aims to be broadly compatible with the `requests` API, although there are a few design differences in places. This documentation outlines places where the API differs... ## Redirects Unlike `requests`, HTTPX does **not follow redirects by default**. We differ in behaviour here [because auto-redirects can easily mask unnecessary network calls being made](https://github.com/encode/httpx/discussions/1785). You can still enable behaviour to automatically follow redirects, but you need to do so explicitly... ```python response = client.get(url, follow_redirects=True) ``` Or', 'arquivo': 'docs/compatibility.md', 'chunk_id': 0, 'titulo': 'Requests Compatibility Guide'}


In [60]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(pergunta, top_k=3):

    if not pergunta.strip():
        print("Digite uma pergunta.")
        return

    if top_k <= 0:
        print("top_k deve ser maior que zero.")
        return

    embedding_pergunta = modelo.encode(
        [pergunta],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        embedding_pergunta,
        embeddings
    )[0]

    indices = scores.argsort()[::-1][:top_k]

    for posicao, indice in enumerate(indices, start=1):

        resultado = chunks[indice]

        print("=" * 70)
        print(f"Resultado {posicao}")
        print(f"Score: {scores[indice]:.4f}")
        print(f"Arquivo: {resultado['arquivo']}")
        print(f"Título: {resultado['titulo']}")
        print(f"Chunk: {resultado['chunk_id']}")
        print()
        print(resultado["texto"])

## Resources

In [61]:
textos = [chunk["texto"] for chunk in chunks]

embeddings = modelo.encode(
    textos,
    normalize_embeddings=True
)

In [62]:
buscar("Como fazer uma requisição GET usando HTTPX?")

Resultado 1
Score: 0.6443
Arquivo: docs/advanced/proxies.md
Título: Sem título
Chunk: 0

HTTPX supports setting up [HTTP proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`. <div align="center"> <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Open_proxy_h2g2bob.svg/480px-Open_proxy_h2g2bob.svg.png"/> <figcaption><em>Diagram of how a proxy works (source: Wikipedia). The left hand side "Internet" blob may be your HTTPX client requesting <code>example.com</code> through a proxy.</em></figcaption> </div> ## HTTP Proxies To route all traffic (HTTP and HTTPS) to a proxy located at `http://localhost:8030`, pass the proxy URL to the client... ```python with httpx.Client(proxy="http://localhost:8030") as client:
Resultado 2
Score: 0.6347
Arquivo: docs/compatibility.md
Título: Requests Compatibility Guide
Chunk: 18

HTTP networking cod

In [65]:
!pip install -q -U google-genai

In [75]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("API Key encontrada:", API_KEY is not None)

API Key encontrada: True


In [76]:
from google import genai

client = genai.Client(api_key=API_KEY)

print("Gemini conectado!")

Gemini conectado!


In [79]:
resposta = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explique em uma frase o que é HTTPX."
)

print(resposta.text)

HTTPX é uma biblioteca Python moderna e completa para fazer requisições HTTP de forma síncrona e assíncrona, com suporte a HTTP/1.1 e HTTP/2.


In [97]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(pergunta, top_k=3):

    if not pergunta.strip():
        print("Digite uma pergunta.")
        return []

    if top_k <= 0:
        print("top_k deve ser maior que zero.")
        return []

    embedding_pergunta = modelo.encode(
        [pergunta],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        embedding_pergunta,
        embeddings
    )[0]

    indices = scores.argsort()[::-1][:top_k]

    resultados = []

    for posicao, indice in enumerate(indices, start=1):

        resultado = chunks[indice].copy()
        resultado["score"] = scores[indice]
        resultado["ranking"] = posicao

        resultados.append(resultado)

    return resultados

In [101]:
def responder_rag(pergunta):

    # 1. Buscar os chunks mais relevantes
    resultados = buscar(pergunta, top_k=3)

    if not resultados:
        return

    # 2. Construir o contexto para o Gemini
    documentos_contexto = ""

    for r in resultados:
        documentos_contexto += f"Arquivo: {r['arquivo']}\n"
        documentos_contexto += f"Título: {r['titulo']}\n"
        documentos_contexto += f"Chunk ID: {r['chunk_id']}\n"
        documentos_contexto += f"Texto: {r['texto']}\n\n"

    # 3. Criar o prompt
    prompt = f"""
Você é um assistente que responde perguntas sobre a documentação do HTTPX.

Use APENAS as informações presentes nos documentos abaixo.

Se a documentação não possuir informação suficiente para responder,
diga que não encontrou informação suficiente na documentação.

DOCUMENTAÇÃO:

{documentos_contexto}

PERGUNTA:

{pergunta}

RESPOSTA:
"""

    # 4. Enviar para o Gemini
    resposta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    # 5. Mostrar a resposta
    print("=" * 70)
    print("RESPOSTA")
    print("=" * 70)
    print(resposta.text)

    # 6. Mostrar as fontes utilizadas
    print("\n")
    print("=" * 70)
    print("FONTES UTILIZADAS")
    print("=" * 70)

    for r in resultados:
        print(f"\nRanking: {r['ranking']}")
        print(f"Score: {r['score']:.4f}")
        print(f"Arquivo: {r['arquivo']}")
        print(f"Título: {r['titulo']}")
        print(f"Chunk: {r['chunk_id']}")

In [102]:
responder_rag("Como fazer uma requisição GET usando HTTPX?")

RESPOSTA
Para fazer uma requisição GET usando HTTPX, primeiro importe a biblioteca e então chame a função `httpx.get()` com a URL desejada.

Exemplo:
```python
import httpx

r = httpx.get('https://httpbin.org/get')
```


FONTES UTILIZADAS

Ranking: 1
Score: 0.6443
Arquivo: docs/advanced/proxies.md
Título: Sem título
Chunk: 0

Ranking: 2
Score: 0.6347
Arquivo: docs/compatibility.md
Título: Requests Compatibility Guide
Chunk: 18

Ranking: 3
Score: 0.5992
Arquivo: docs/quickstart.md
Título: QuickStart
Chunk: 0


In [106]:
responder_rag("Como o HTTPX trabalha com proxies?")

RESPOSTA
O HTTPX suporta a configuração de proxies HTTP através do parâmetro `proxy`, que pode ser passado na inicialização do cliente ou em funções de API de nível superior, como `httpx.get(..., proxy=...)`.

Para rotear todo o tráfego (HTTP e HTTPS) para um proxy, a URL do proxy deve ser passada para o cliente, por exemplo: `with httpx.Client(proxy="http://localhost:8030") as client:`. No caso de tráfego HTTPS através de um proxy HTTP, o cliente "atualiza" a conexão para HTTPS realizando o handshake TLS com o servidor sobre a conexão TCP fornecida pelo proxy.

Além dos proxies HTTP, `httpcore` também suporta proxies usando o protocolo SOCKS. Esta é uma funcionalidade opcional que requer a instalação de uma biblioteca de terceiros adicional antes do uso.


FONTES UTILIZADAS

Ranking: 1
Score: 0.7696
Arquivo: docs/advanced/proxies.md
Título: Sem título
Chunk: 0

Ranking: 2
Score: 0.6613
Arquivo: docs/compatibility.md
Título: Requests Compatibility Guide
Chunk: 18

Ranking: 3
Score: 0.6

In [107]:
responder_rag("Quem ganhou a Copa do Mundo de 2022?")

RESPOSTA
Não encontrei informação suficiente na documentação para responder à sua pergunta sobre quem ganhou a Copa do Mundo de 2022.


FONTES UTILIZADAS

Ranking: 1
Score: 0.1377
Arquivo: docs/quickstart.md
Título: QuickStart
Chunk: 24

Ranking: 2
Score: 0.1246
Arquivo: docs/advanced/extensions.md
Título: Extensions
Chunk: 4

Ranking: 3
Score: 0.1214
Arquivo: docs/troubleshooting.md
Título: Troubleshooting
Chunk: 4
